# Optimal Pandas: Vectorization and Parallelization

This notebook is a **from-scratch to intermediate/advanced** guide to using **Pandas**
efficiently: vectorization, avoiding slow patterns, and when/how to parallelize
operations (including **Pandarallel**).

We use a **large dataframe of FX-style random prices** (EURUSD, GBPUSD, AUDUSD)
generated from a random walk starting near current reference levels (~1.15, 1.33, 0.71)
as the main study case. We **time** operations to compare optimal vs non-optimal
approaches.

### Contents

- [1. Why Vectorization and Optimal Pandas Matter](#1-why-vectorization-and-optimal-pandas-matter)
- [2. Building the Study Dataset: Random-Walk FX Prices](#2-building-the-study-dataset-random-walk-fx-prices)
- [3. Loops vs Vectorized Operations (Timing Baseline)](#3-loops-vs-vectorized-operations-timing-baseline)
- [4. Column-Wise Operations and Avoiding Iteration](#4-column-wise-operations-and-avoiding-iteration)
- [5. GroupBy, Aggregation, and Transform](#5-groupby-aggregation-and-transform)
- [6. Rolling and Window Operations](#6-rolling-and-window-operations)
- [7. Parallelization with Pandarallel](#7-parallelization-with-pandarallel)
- [8. When to Parallelize and Best Practices](#8-when-to-parallelize-and-best-practices)

---

## 1. Why Vectorization and Optimal Pandas Matter

Pandas is built on **NumPy**, so operations that work on whole arrays (columns/series)
run in compiled C code and are much faster than Python loops over rows. **Vectorization**
means expressing your logic as array operations instead of row-by-row loops.

- **Vectorized**: `df['col'] * 2`, `df['a'] + df['b']`, `df.groupby('x').sum()`
- **Slow**: `for i, row in df.iterrows(): ...`

Some workflows can also be **parallelized** (e.g. `apply` over groups, or independent
transformations) using libraries like **Pandarallel**, which use multiple CPU cores.
We'll compare timings to see when parallelization helps.

---

## 2. Building the Study Dataset: Random-Walk FX Prices

We create a large DataFrame with three symbols (**EURUSD**, **GBPUSD**, **AUDUSD**)
and a price series per symbol generated by a **random walk** starting from reference
levels (~1.15, 1.33, 0.71). This gives realistic-looking price data for timing tests.

In [32]:
import numpy as np
import pandas as pd
from time import perf_counter

# Reference levels (approx current FX levels)
REF = {'EURUSD': 1.15, 'GBPUSD': 1.33, 'AUDUSD': 0.71,
      #'USDJPY': 105., 'USDCAD': 1.25, 'NZDUSD': 0.65,
      #'XAGUSD': 20., 'XAUUSD': 1700., 'BTCUSD': 50000.
}
N_ROWS = 500_000  # large enough to see timing differences

def random_walk(start: float, n: int, vol: float = 0.0005, seed: int = 42) -> np.ndarray:
    """Simple log-normal style random walk: start * exp(cumsum of returns)."""
    rng = np.random.default_rng(seed)
    returns = rng.standard_normal(n) * vol
    series = start * np.exp(np.cumsum(returns))
    digits = int(6 - np.log10(start))
    return series.round(digits)

# Build long-format DataFrame (vectorized: one block per symbol, then concat)
pieces = []
for symbol, start in REF.items():
    price = random_walk(start, N_ROWS, vol=0.0003)
    pieces.append(pd.DataFrame({'symbol': symbol, 'timestamp': np.arange(N_ROWS), 'price': price}))
df = pd.concat(pieces, ignore_index=True)
print(df.head(10))
print(f"\nShape: {df.shape}, memory: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

   symbol  timestamp    price
0  EURUSD          0  1.15011
1  EURUSD          1  1.14975
2  EURUSD          2  1.15001
3  EURUSD          3  1.15033
4  EURUSD          4  1.14966
5  EURUSD          5  1.14921
6  EURUSD          6  1.14925
7  EURUSD          7  1.14914
8  EURUSD          8  1.14914
9  EURUSD          9  1.14884

Shape: (1500000, 3), memory: 106.50 MB


In [33]:
# Quick sanity check: mean price per symbol should be near reference
print(df.groupby('symbol')['price'].agg(['mean', 'min', 'max']))

            mean       min      max
symbol                             
AUDUSD  0.681428  0.603276  0.78162
EURUSD  1.103722  0.977140  1.26600
GBPUSD  1.276479  1.130080  1.46416


---

## 3. Loops vs Vectorized Operations (Timing Baseline)

We compare:
- **A**: Python loop over rows (e.g. `iterrows` or `itertuples`) to compute a new column.
- **B**: Same logic expressed as vectorized operations on columns.

Example: `returns = (price - prev_price) / prev_price` approximated by a simple
difference-based proxy for one symbol.

In [34]:
# Subset for slow methods so we don't wait forever
df_small = df.head(50_000).copy()

# --- A: Loop (iterrows) — avoid in production
t0 = perf_counter()
ret_loop = []
prev = np.nan
for _, row in df_small.iterrows():
    if np.isnan(prev):
        ret_loop.append(np.nan)
    else:
        ret_loop.append((row['price'] - prev) / prev)
    prev = row['price']
df_small['ret_loop'] = ret_loop
t_loop = perf_counter() - t0
print(f"iterrows loop: {t_loop:.3f} s")

iterrows loop: 0.689 s


In [35]:
# --- B: Vectorized (per-symbol pct_change on full df)
t0 = perf_counter()
df['ret_vec'] = df.groupby('symbol')['price'].pct_change()
t_vec = perf_counter() - t0
print(f"Vectorized pct_change (full df): {t_vec:.3f} s")

# Compare on the same 50k rows
df_50 = df.head(50_000).copy()
df_50['ret_vec'] = df_50.groupby('symbol')['price'].pct_change()
t0 = perf_counter()
df_50['ret_vec'] = df_50.groupby('symbol')['price'].pct_change()
t_vec50 = perf_counter() - t0
print(f"Vectorized on 50k rows: {t_vec50:.3f} s")
print(f"Speedup vs iterrows (50k): ~{t_loop / t_vec50:.0f}x")

Vectorized pct_change (full df): 0.091 s
Vectorized on 50k rows: 0.003 s
Speedup vs iterrows (50k): ~237x


---

## 4. Column-Wise Operations and Avoiding Iteration

Use built-in column operations instead of `apply` when possible: arithmetic, `np.where`,
`clip`, `rolling`, etc. Reserve `apply` for row-wise logic that has no vectorized
equivalent, and prefer `axis=1` only when necessary.

In [36]:
# Example: log return = log(price / prev_price)
# Vectorized (fast)
t0 = perf_counter()
df['log_ret'] = np.log(df.groupby('symbol')['price'].transform(lambda x: x / x.shift(1)))
t_vec = perf_counter() - t0
print(f"Vectorized log return: {t_vec:.3f} s")

# Same with apply (slower)
t0 = perf_counter()
df['log_ret_apply'] = df.groupby('symbol')['price'].apply(lambda x: np.log(x / x.shift(1))).droplevel(0)
t_apply = perf_counter() - t0
print(f"apply-based log return: {t_apply:.3f} s")
print(f"Vectorized is ~{t_apply / t_vec:.1f}x faster")
print(df[['symbol', 'price', 'ret_vec', 'log_ret']].head(8))

Vectorized log return: 0.163 s
apply-based log return: 0.241 s
Vectorized is ~1.5x faster
   symbol    price   ret_vec   log_ret
0  EURUSD  1.15011       NaN       NaN
1  EURUSD  1.14975 -0.000313 -0.000313
2  EURUSD  1.15001  0.000226  0.000226
3  EURUSD  1.15033  0.000278  0.000278
4  EURUSD  1.14966 -0.000582 -0.000583
5  EURUSD  1.14921 -0.000391 -0.000391
6  EURUSD  1.14925  0.000035  0.000035
7  EURUSD  1.14914 -0.000096 -0.000096


---

## 5. GroupBy, Aggregation, and Transform

GroupBy is already optimized: use `.agg()`, `.sum()`, `.mean()`, or `.transform()` 
instead of looping over groups. We time aggregations and transform on our price data.

In [37]:
# Aggregation: one row per symbol
t0 = perf_counter()
agg = df.groupby('symbol').agg(mean_price=('price', 'mean'), std_price=('price', 'std'), count=('price', 'count'))
t_agg = perf_counter() - t0
print("Aggregation (mean, std, count):", agg)
print(f"Time: {t_agg:.3f} s\n")

# Transform: broadcast group stats back to each row
t0 = perf_counter()
df['price_minus_mean'] = df['price'] - df.groupby('symbol')['price'].transform('mean')
t_tr = perf_counter() - t0
print(f"Transform (price - group mean): {t_tr:.3f} s")
print(df[['symbol', 'price', 'price_minus_mean']].head(6))

Aggregation (mean, std, count):         mean_price  std_price   count
symbol                               
AUDUSD    0.681428   0.037647  500000
EURUSD    1.103722   0.060977  500000
GBPUSD    1.276479   0.070522  500000
Time: 0.057 s

Transform (price - group mean): 0.050 s
   symbol    price  price_minus_mean
0  EURUSD  1.15011          0.046388
1  EURUSD  1.14975          0.046028
2  EURUSD  1.15001          0.046288
3  EURUSD  1.15033          0.046608
4  EURUSD  1.14966          0.045938
5  EURUSD  1.14921          0.045488


---

## 6. Rolling and Window Operations

Rolling mean, std, etc. are vectorized per series. Apply them inside `groupby` so
each symbol has its own window (no cross-contamination across symbols).

In [38]:
WINDOW = 20
t0 = perf_counter()
df['rolling_mean'] = df.groupby('symbol')['price'].transform(lambda s: s.rolling(WINDOW, min_periods=1).mean())
df['rolling_std']  = df.groupby('symbol')['price'].transform(lambda s: s.rolling(WINDOW, min_periods=1).std())
t_roll = perf_counter() - t0
print(f"Rolling mean & std (window={WINDOW}): {t_roll:.3f} s")
print(df[['symbol', 'price', 'rolling_mean', 'rolling_std']].head(25))

Rolling mean & std (window=20): 0.318 s
    symbol    price  rolling_mean  rolling_std
0   EURUSD  1.15011      1.150110          NaN
1   EURUSD  1.14975      1.149930     0.000255
2   EURUSD  1.15001      1.149957     0.000186
3   EURUSD  1.15033      1.150050     0.000241
4   EURUSD  1.14966      1.149972     0.000272
5   EURUSD  1.14921      1.149845     0.000395
6   EURUSD  1.14925      1.149760     0.000425
7   EURUSD  1.14914      1.149682     0.000450
8   EURUSD  1.14914      1.149622     0.000458
9   EURUSD  1.14884      1.149544     0.000498
10  EURUSD  1.14915      1.149508     0.000487
11  EURUSD  1.14941      1.149500     0.000465
12  EURUSD  1.14944      1.149495     0.000446
13  EURUSD  1.14983      1.149519     0.000438
14  EURUSD  1.14999      1.149551     0.000439
15  EURUSD  1.14969      1.149559     0.000425
16  EURUSD  1.14982      1.149575     0.000417
17  EURUSD  1.14949      1.149570     0.000405
18  EURUSD  1.14979      1.149582     0.000397
19  EURUSD  1.14977 

---

## 7. Parallelization with Pandarallel

**Pandarallel** uses multiple CPU cores to parallelize `apply`, `map`, `groupby().apply()`,
etc. Install with: `pip install pandarallel`. Initialize once with `pandarallel.initialize()` (optional: set `nb_workers`, `progress_bar=True`).

Best gains when:
- You have many groups or rows (hundreds of thousands to millions of rows).
- Your per-row/per-group function is not simply expressible with vectorized Pandas/NumPy.
- The function does substantial work (tens of milliseconds or more per group).

On **Windows in particular**, process-based parallelism has extra overhead: the OS
cannot `fork` the current process, so each worker must start a fresh Python interpreter
and import your code from scratch. Data and functions are passed via pickling,
which adds serialization cost. This makes the fixed overhead of Pandarallel higher
on Windows than on Unix-like systems, so you need *even larger* workloads before you
see a speedup.

In [39]:
# Install if needed: pip install pandarallel
try:
    from pandarallel import pandarallel
    pandarallel.initialize(nb_workers=4, progress_bar=False)
    HAS_PANDARALLEL = True
except ImportError:
    HAS_PANDARALLEL = False
    print("Pandarallel not installed. Run: pip install pandarallel")

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/



Originally, in **this notebook's FX example**, we only had **three symbols** (EURUSD, GBPUSD, AUDUSD) and a relatively light per-group computation (a z-score of price). The timing typically
shows **serial `groupby().apply()` is faster than Pandarallel's `parallel_apply`** in such case.
That is expected: the overhead of starting worker processes and moving data between
processes is larger than the small amount of per-group work.

That makes a good example of a case where **Pandarallel does *not* help much (and can
be slower)**, and reinforces the rule: **vectorize first; parallelize only when you
have many heavy, independent pieces of work.**

We compare standard `groupby().apply()` vs Pandarallel's parallel apply on the same task.

In [42]:
# Example: per-group "normalized price" (z-score within symbol) using apply
# We write the formula inline so workers don't need to import a named function.

# Serial apply (standard pandas)
t0 = perf_counter()
ser_serial_df = df.groupby('symbol').apply(
    lambda g: (g['price'] - g['price'].mean()) / g['price'].std()
)
ser_serial = ser_serial_df.droplevel(0)
t_serial = perf_counter() - t0
print(f"Serial groupby.apply(zscore): {t_serial:.3f} s")

if HAS_PANDARALLEL:
    # Pandarallel patches DataFrameGroupBy; same logic, but in parallel
    t0 = perf_counter()
    ser_parallel_df = df.groupby('symbol').parallel_apply(
        lambda g: (g['price'] - g['price'].mean()) / g['price'].std()
    )
    ser_parallel = ser_parallel_df.droplevel(0)
    t_parallel = perf_counter() - t0
    print(f"Pandarallel groupby.parallel_apply(zscore): {t_parallel:.3f} s")
    print(f"Speedup: {t_serial / t_parallel:.2f}x")
else:
    print("(Skipping parallel timing; pandarallel not available)")

C:\Users\GSL\AppData\Local\Temp\ipykernel_52356\2949960042.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ser_serial_df = df.groupby('symbol').apply(


Serial groupby.apply(zscore): 0.251 s
Pandarallel groupby.parallel_apply(zscore): 0.948 s
Speedup: 0.27x


---

## 8. When to Parallelize and Best Practices

- **Prefer vectorization** over both loops and parallel apply: built-in ops (arithmetic, `groupby`, `rolling`) are already fast and single-threaded in C.
- **Use Pandarallel** when you must use `apply` (no vectorized alternative) and the DataFrame/group size and per-row work justify process overhead.
- **Avoid** `iterrows()` and row-wise Python loops for bulk math; use `itertuples()` only if you truly need row-by-row and cannot vectorize.
- **Dtypes**: use `category` for repeated strings (e.g. `symbol`), and downcast numerics where possible to reduce memory and sometimes speed.
- **Chunking**: for very large data, process in chunks (e.g. `pd.read_csv(..., chunksize=...)`) and combine results.

In [43]:
# Optional: dtype optimization
df_opt = df.copy()
df_opt['symbol'] = df_opt['symbol'].astype('category')
# Downcast float64 to float32 if precision allows
for col in ['price', 'ret_vec', 'rolling_mean']:
    if col in df_opt.columns:
        df_opt[col] = pd.to_numeric(df_opt[col], downcast='float')
print("Dtypes after optimization:")
print(df_opt.dtypes)
print(f"\nMemory original: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Memory optimized: {df_opt.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Dtypes after optimization:
symbol              category
timestamp              int64
price                float32
ret_vec              float32
log_ret              float64
log_ret_apply        float64
price_minus_mean     float64
rolling_mean         float32
rolling_std          float64
dtype: object

Memory original: 178.5 MB
Memory optimized: 79.5 MB


### Summary

| Approach | When to use |
|----------|-------------|
| Vectorized column/series ops | Default: arithmetic, `pct_change`, `rolling`, `groupby().agg/transform` |
| `apply` | Only when no vectorized alternative exists |
| Pandarallel | Heavy `apply` over many rows or many groups |
| Loops / `iterrows` | Avoid for bulk numeric work; use only for one-off or tiny data |
| Dtype/category/downcast | Reduce memory and sometimes improve speed on large DataFrames |